In [ ]:
%load_ext cudf.pandas

In [ ]:
%%time
### cell 0 ###

import pandas as pd

# load just the datasets q01 needs.
factor = 10
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/lineitem.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    print(df.columns)
    df["L_SHIPDATE"] = pd.to_datetime(df.L_SHIPDATE, format="%Y-%m-%d")
    df["L_RECEIPTDATE"] = pd.to_datetime(df.L_RECEIPTDATE, format="%Y-%m-%d")
    df["L_COMMITDATE"] = pd.to_datetime(df.L_COMMITDATE, format="%Y-%m-%d")
    return df
lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%time
### cell 0 ###

threshold = '1998-09-02'
cols = [
    'L_QUANTITY', 'L_EXTENDEDPRICE', 'L_DISCOUNT', 'L_TAX',
    'L_RETURNFLAG', 'L_LINESTATUS', 'L_SHIPDATE', 'L_ORDERKEY'
]

# 1) Single .loc with bracket indexing for projection & filter
lf = lineitem.loc[lineitem['L_SHIPDATE'] <= threshold, cols]

In [ ]:
%%time
### cell 1 ###

lf = lf.assign(
    AVG_QTY=lf['L_QUANTITY'],
    AVG_PRICE=lf['L_EXTENDEDPRICE'],
    DISC_PRICE=lf['L_EXTENDEDPRICE'] * (1 - lf['L_DISCOUNT']),
    CHARGE=(
        lf['L_EXTENDEDPRICE'] * (1 - lf['L_DISCOUNT']) * (1 + lf['L_TAX'])
    )
)

# 3) GPU-friendly groupby & aggregation
total = lf.groupby(
    ['L_RETURNFLAG', 'L_LINESTATUS'], as_index=False
).agg({
    'L_QUANTITY': 'sum',
    'L_EXTENDEDPRICE': 'sum',
    'DISC_PRICE': 'sum',
    'CHARGE': 'sum',
    'AVG_QTY': 'mean',
    'AVG_PRICE': 'mean',
    'L_DISCOUNT': 'mean',
    'L_ORDERKEY': 'count'
})